In [ ]:
import digitalhub as dh

dh.get_current_profile()

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Carica le variabili d'ambiente dal file .env


# Inizializza il client (assicurati di aver impostato la variabile d'ambiente OPENAI_API_KEY)
client = OpenAI()


In [3]:
from pathlib import Path
from contextlib import ExitStack

folder = Path("./file_to_load")
all_files = sorted(p for p in folder.iterdir() if p.is_file())
file_names = [p.name for p in all_files]
print("File trovati:", file_names)

# Only for test: first  5 PDF 
pdf_paths = [p for p in all_files if p.suffix.lower() == ".pdf"][:5]
if not pdf_paths:
    raise FileNotFoundError(f"Nessun PDF trovato in {folder.resolve()}")

print("PDF selezionati:", [p.name for p in pdf_paths])



File trovati: ['Modulo1_Lezione1-1_Bibliografia.pdf', 'Modulo1_Lezione1-1_Caso1.pdf', 'Modulo1_Lezione1-1_Caso2.pdf', 'Modulo1_Lezione1-1_Guida.pdf', 'Modulo1_Lezione1-1_Materiale.pdf']
PDF selezionati: ['Modulo1_Lezione1-1_Bibliografia.pdf', 'Modulo1_Lezione1-1_Caso1.pdf', 'Modulo1_Lezione1-1_Caso2.pdf', 'Modulo1_Lezione1-1_Guida.pdf', 'Modulo1_Lezione1-1_Materiale.pdf']


In [4]:
def get_or_create_vector_store(client, name="spg-Core"):
    for store in client.vector_stores.list():
        if store.name == name:
            print(f"Vector store esistente: {store.id}")
            return store

    store = client.vector_stores.create(name=name)
    print(f"Vector store creato: {store.id}")
    return store


vector_store = get_or_create_vector_store(client)
vector_store_id = vector_store.id

Vector store esistente: vs_6af197814c448191ac0823f3d5937996


In [5]:
# carica i file nel vector store e li indicizza"
# se il file é già presente con lo stesso contenuto, non viene ricaricato per evitare duplicazioni:
# lo store di openAI non gestire duplicazioni di file, quindi se un file viene caricato due volte, verrà indicizzato altrettante volte

from pathlib import Path
import hashlib

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_new_files(client, vector_store, folder="./file_to_load", max_file_count=5):
    pdf_paths = sorted(
        p for p in Path(folder).iterdir()
        if p.is_file() and p.suffix.lower() == ".pdf"
    )[:max_file_count]

    # .
    loaded_hashes = set()
    page = client.vector_stores.files.list(
        vector_store_id=vector_store.id,
        limit=100,
    )
    while True:
        for file in page.data:
            attributes = file.attributes or {}
            file_hash = attributes.get("sha256")
            if file_hash:
                loaded_hashes.add(file_hash)

        if not page.has_next_page():
            break
        page = page.get_next_page()

    loaded = []
    skipped = []

    for path in pdf_paths:
        file_hash = sha256_file(path)

        if file_hash in loaded_hashes:
            skipped.append(path.name)
            print(f"Già presente, salto: {path.name}")
            continue

        with path.open("rb") as stream:
            uploaded = client.files.create(file=stream, purpose="assistants")

        indexed = client.vector_stores.files.create_and_poll(
            vector_store_id=vector_store.id,
            file_id=uploaded.id,
            attributes={"sha256": file_hash},
        )

        if indexed.status != "completed":
            raise RuntimeError(
                f"Indicizzazione non completata per {path.name}: {indexed.status}"
            )

        loaded_hashes.add(file_hash)
        loaded.append(path.name)
        print(f"Caricato e indicizzato: {path.name}")

    return {"caricati": loaded, "saltati": skipped  }


res = load_new_files(client, vector_store)
print(res)



Caricato e indicizzato: Modulo1_Lezione1-1_Bibliografia.pdf
Caricato e indicizzato: Modulo1_Lezione1-1_Caso1.pdf
Caricato e indicizzato: Modulo1_Lezione1-1_Caso2.pdf
Caricato e indicizzato: Modulo1_Lezione1-1_Guida.pdf
Caricato e indicizzato: Modulo1_Lezione1-1_Materiale.pdf
{'caricati': ['Modulo1_Lezione1-1_Bibliografia.pdf', 'Modulo1_Lezione1-1_Caso1.pdf', 'Modulo1_Lezione1-1_Caso2.pdf', 'Modulo1_Lezione1-1_Guida.pdf', 'Modulo1_Lezione1-1_Materiale.pdf'], 'saltati': []}


In [10]:
prompt = (
    "Sei un assistente di ricerca. Rispondi alle domande basandoti esclusivamente sui file caricati. "
    "Se non trovi la risposta nei file, dichiaralo chiaramente."
)
question = "quali file stai considerando per rispondere a questa domanda?"
response = client.responses.create(
        model="gpt-4o",
        instructions=prompt,
        input=question,
        tools=[
            {
                "type": "file_search",
                "vector_store_ids": [vector_store.id],
            }
        ],

    )
print("ID della risposta:", response.id)
print("Stato della risposta:", response.status)
print("risposta:", response.output_text)


ID della risposta: resp_08072eaf1c1a8698006af1d6646cd087d2bce14300579990f6
Stato della risposta: completed
risposta: Sto considerando il file che hai caricato, ma non ci sono dettagli specifici forniti sulla sua natura o contenuto al momento. Posso rispondere a domande basate su informazioni che trovo nel documento, quindi ti invito a formulare una domanda per iniziare la ricerca.


# File search gerarchico

Caricamento dei file con classificazione basata sulla directory (deterministico)

file_to_load/

├── core/<br>
├── internal/<br>
└── external/<br>

## PFD -> MD 
Per ciascun file pdf nelle sottocartelle viene esportato il solo contento testuale in rispettivo file md


In [ ]:
from pathlib import Path
import base64
import pymupdf4llm

# ricerca semantica:
# mettere chunk  a paragrafi 
# inserire nel vector store (indicizzazione)
# effettuare la query di somilianza sul file
# 


#usa openai per fare tutto il processo

def exportToMarkdown(input_path, output_dir="./exported"):
    for pdf_path in input_path.iterdir():
        if not pdf_path.is_file() or pdf_path.suffix.lower() != ".pdf":
            continue
        try:
            markdown = pymupdf4llm.to_markdown(
                str(pdf_path),
                write_images=False,
                embed_images=False,
            )

            markdown_path = pdf_path.with_suffix(".md")
            markdown_path.write_text(markdown, encoding="utf-8")
            print(f"Created: {markdown_path.name}")

        except Exception as error:
            print(f"Error processing {pdf_path.name}: {error}")






root = Path("./file_to_load")
classi = {"core", "nearby", "external"}

for folder in classi:
    folder_path = root / folder
    exportToMarkdown(folder_path)





def riassumi_pdf(client, path):
    pdf_data = base64.b64encode(path.read_bytes()).decode("ascii")

    response = client.responses.create(
        model="gpt-4o",
        input=[{
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": path.name,
                    "file_data": f"data:application/pdf;base64,{pdf_data}",
                    "detail": "low",
                },
                {
                    "type": "input_text",
                    "text": (
                        "Riassumi il documento in italiano in massimo 450 caratteri. "
                        "Indica argomento e contenuti principali, senza inventare."
                    ),
                },
            ],
        }],
    )
    return response.output_text.strip()[:512]



# for path in sorted(root.rglob("*.pdf")):
#     percorso_relativo = path.relative_to(root)

#     if len(percorso_relativo.parts) < 2 or percorso_relativo.parts[0] not in classi:
#         raise ValueError(f"Cartella di classificazione non valida: {path}")

#     gruppo = percorso_relativo.parts[0]
#     riassunto = riassumi_pdf(client, path)

#     with path.open("rb") as stream:
#         uploaded = client.files.create(file=stream, purpose="assistants")

#     indexed = client.vector_stores.files.create_and_poll(
#         vector_store_id=vector_store.id,
#         file_id=uploaded.id,
#         attributes={
#             "group": gruppo,
#             "summary": riassunto,
#         },
#     )

#     if indexed.status != "completed":
#         raise RuntimeError(f"Indicizzazione fallita per {path.name}: {indexed.status}")

#     print(f"Caricato: {path.name} — {gruppo}")


=== Document parser messages ===
Created: Modulo6_Lezione6-4-2_Bibliografia.md


Traceback (most recent call last):
  File "/home/cappelle/python3.12/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3823, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_344035/102174118.py", line 62, in <module>
    exportToMarkdown(folder_path)
  File "/tmp/ipykernel_344035/102174118.py", line 11, in exportToMarkdown
    markdown = pymupdf4llm.to_markdown(
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cappelle/python3.12/lib/python3.12/site-packages/pymupdf4llm/__init__.py", line 235, in to_markdown
    return _layout_to_markdown(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cappelle/python3.12/lib/python3.12/site-packages/pymupdf4llm/__init__.py", line 85, in _layout_to_markdown
    parsed_doc = pymupdf4llm.helpers.document_layout.parse_document(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/cappelle/python3.12/lib/python3.12/site-packages/pymu

KeyboardInterrupt: 

In [ ]:
response = client.responses.create(
    model="gpt-4o",
    input=question,
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
    }],
)

metadati = {}
page = client.vector_stores.files.list(
    vector_store_id=vector_store.id,
    limit=100,
)

while True:
    for file in page.data:
        metadati[file.id] = file.attributes or {}

    if not page.has_next_page():
        break
    page = page.get_next_page()

citazioni = {}
for item in response.output:
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                for annotation in content.annotations:
                    if annotation.type == "file_citation":
                        citazioni[annotation.file_id] = annotation.filename

print(response.output_text)
print("\nFonti:")
for file_id, nome in citazioni.items():
    info = metadati.get(file_id, {})
    print(f"- File: {nome}")
    print(f"  Gerarchia: {info.get('group', 'non specificata')}")
    print(f"  Riassunto: {info.get('summary', 'non disponibile')}")